# Atividade 1 — Segmentação de clientes com aprendizado não supervisionado

**Objetivo:** identificar perfis de compra e traduzir os agrupamentos em ações de marketing e finanças.

**Base:** Chen, D. (2015), [UCI Online Retail](https://doi.org/10.24432/C5BW33), CC BY 4.0. Transações históricas de dezembro de 2010 a dezembro de 2011; valores em GBP. A base inclui atacadistas.

Este notebook atende aos requisitos detalhados no PDF: comparação de K-Means, hierárquico e GMM, visualizações, interpretação e relatório executivo. A opção GMM substitui DBSCAN conforme permitido. O código reutilizável está em `atividade1.py`.

Execute as células em ordem usando o ambiente com `requirements.txt`. O download ocorre somente se os dados não estiverem em `dados/`. A execução não depende dos resultados já salvos e pode demorar alguns minutos.

**Versão pública:** as saídas do notebook são geradas ao executar as células. Os arquivos completos permanecem disponíveis localmente.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown, Image
from atividade1 import executar, ROOT

out = ROOT / 'resultados'
rfm, perfil, comparacao, resumo = executar()

def grafico(nome):
    display(pio.read_json(out / f'{nome}.json'))

pd.set_option('display.max_columns', 20)

## 1. Auditoria e construção das variáveis

Foram removidas duplicatas exatas e, sequencialmente, linhas com clientes ausentes, campos inválidos, cancelamentos, quantidades e preços não positivos. Cada exclusão é contada somente uma vez. Não foi realizada imputação da identidade dos clientes; descrições ausentes não invalidam a transação.

- **Recency:** dias entre a última compra e o dia seguinte à data máxima da base.
- **Frequency:** quantidade de notas fiscais distintas (uma nota pode conter várias linhas).
- **Monetary:** soma de quantidade × preço em libras.

Foram aplicadas a transformação `log1p` e a padronização às três variáveis, reduzindo assimetria e equilibrando a distância euclidiana. Ticket médio, diversidade de produtos e tempo observado são características complementares; não aumentam o peso de valor/frequência no ajuste. Clientes extremos são preservados. A receita é bruta das compras válidas identificadas, sem compensação de devoluções.

In [ ]:
auditoria = json.loads((out / 'auditoria.json').read_text())
display(pd.Series(auditoria, name='Valor').to_frame())
display(rfm[['Recency', 'Frequency', 'Monetary', 'TicketMedio', 'ProdutosDistintos', 'TempoObservadoDias']].describe().round(2))
exclusoes = ['duplicatas', 'sem_cliente', 'campos_invalidos', 'cancelamentos', 'quantidade_nao_positiva', 'preco_nao_positivo']
assert sum(auditoria[k] for k in exclusoes) + auditoria['linhas_validas'] == auditoria['linhas_originais']
assert np.isclose(rfm.Monetary.sum(), auditoria['receita_bruta'])
assert rfm.index.is_unique

## 2. K-Means: número de grupos, convergência e estabilidade

Busca entre 2 e 8 grupos, semente 42, 20 inicializações e limite de 500 iterações. O cotovelo é estimado pela maior distância da curva de inércia normalizada à reta entre seus extremos. É uma heurística exploratória; a escolha operacional usa maior silhouette, sujeita a pelo menos 2% dos clientes em cada grupo.

A estabilidade é avaliada com dez sementes e ARI entre as partições, independente da numeração dos clusters. ARI próximo de 1 indica concordância. A convergência é verificada pelas iterações e inércias; atingir o limite de iterações é tratado conservadoramente como falta de convergência.

In [ ]:
grafico('cotovelo')
grafico('silhouette')
display(comparacao[comparacao.familia.eq('KMeans')][['k','inercia','silhouette','iteracoes','convergiu']])
display(pd.read_csv(out / 'estabilidade_kmeans.csv'))
display(Markdown(f"Cotovelo: **k={resumo['k_cotovelo']}**. Melhor K-Means elegível: **{resumo['kmeans']}**. ARI médio entre sementes: **{resumo['ari_medio']:.4f}**. A divergência entre cotovelo e silhouette representa objetivos distintos: compactação e separação."))

## 3. Agrupamento hierárquico

Foram comparados os métodos `ward`, `complete` e `average`, de 2 a 8 grupos. Ward favorece fusões que aumentam menos a variância interna; complete usa a maior distância entre membros de dois grupos; average usa a distância média.

O corte operacional maximiza silhouette entre as configurações que respeitam o tamanho mínimo. A árvore usa todos os clientes; o dendrograma trunca apenas a exibição aos últimos 30 ramos. A altura do corte fica entre as fusões que delimitam o número selecionado de grupos. Saltos verticais maiores sinalizam fusões de grupos mais distantes, sem provar um número verdadeiro de segmentos.

In [ ]:
display(comparacao[comparacao.familia.eq('Hierarquico')][['metodo','k','silhouette','davies_bouldin','menor_grupo']])
display(Image(filename=str(out / 'dendrograma.png')))
display(pd.Series(resumo['hierarquia'], name='Corte selecionado').to_frame())

## 4. Gaussian Mixture Models e incerteza

Foram avaliadas configurações com 1 a 8 componentes e covariâncias `full` e `diag`. BIC/AIC menores indicam melhor compromisso entre ajuste da densidade e complexidade dentro dessa busca. Um componente é incluído como referência, mas não forma uma segmentação multigrupo.

O melhor BIC global é reportado mesmo se não atender à restrição operacional. Probabilidades de pertencimento permitem reconhecer sobreposição: probabilidade máxima abaixo de 70% é um indicador heurístico de ambiguidade, não uma probabilidade de recompra. Uma mistura pode apresentar bom BIC e silhouette baixa, pois otimiza densidade, não separação geométrica.

In [ ]:
grafico('gmm')
probabilidades = pd.read_csv(out / 'probabilidades_gmm.csv', dtype={'CustomerID':str})
display(probabilidades.drop(columns='CustomerID').head())
display(Markdown(f"Mínimo BIC: **{resumo['gmm_bic_global']}**; mínimo AIC: **{resumo['gmm_aic_global']}**. Clientes com pertinência incerta: **{resumo['gmm_incertos']}**. Se o mínimo ocorrer em oito componentes, a busca atingiu sua fronteira: não se pode afirmar um ótimo global."))
colunas_prob = [c for c in probabilidades if c.startswith('componente_')]
assert np.allclose(probabilidades[colunas_prob].sum(axis=1), 1)

## 5. Comparação e escolha final

Silhouette maior, Davies–Bouldin menor e Calinski–Harabasz maior são desejáveis. Todos são calculados sobre os mesmos clientes no espaço RFM transformado, sem amostragem. O tempo mede ajuste/atribuição, sem incluir métricas.

A seleção usa modelos convergidos com grupos de pelo menos 2% dos clientes. Foi selecionado um candidato de cada abordagem pelo maior silhouette (K-Means e cada linkage) ou menor BIC (cada tipo de covariância GMM). Entre candidatos, vence a menor média dos três rankings internos; desempate pelo maior silhouette. O limiar de tamanho é uma decisão operacional explícita, não propriedade estatística da base. Partições inelegíveis permanecem disponíveis para inspeção.

In [ ]:
display(pd.read_csv(out / 'selecao_modelos.csv'))
display(comparacao.round(4))
display(Markdown(f"**Modelo final: {resumo['escolhido']}**. Métricas internas favorecem determinadas geometrias; esta seleção não mede retorno financeiro ou capacidade de generalizar para outro período."))

## 6. PCA em duas e três dimensões

A clusterização usa RFM padronizado completo; PCA serve à visualização. A variância preservada foi informada explicitamente. Os símbolos pretos são médias de cada grupo projetadas; no hierárquico e GMM, não representam necessariamente parâmetros do modelo.

A tabela mostra os coeficientes dos autovetores. A magnitude indica contribuição para o eixo e o sinal indica direção relativa, podendo ser invertido sem alterar a solução.

In [ ]:
cargas = pd.read_csv(out / 'pca_cargas.csv', index_col=0)
display(cargas.round(4))
display(pd.DataFrame({'Variancia':resumo['pca_variancia'], 'Acumulada':np.cumsum(resumo['pca_variancia'])}, index=['PC1','PC2','PC3']))
for pc in cargas:
    descricao = ', '.join(f'{f} ({cargas.loc[f, pc]:+.3f})' for f in cargas[pc].abs().sort_values(ascending=False).index)
    display(Markdown(f'**{pc}**, em ordem de magnitude: {descricao}.'))
grafico('pca2d')
grafico('pca3d')

Na base fornecida, PC1 combina frequência e valor positivos com recência negativa: maior pontuação corresponde a compras mais frequentes, valiosas e recentes. PC2 é dominado pela recência positiva, destacando tempo sem comprar. PC3 contrasta frequência e valor, distinguindo padrões de pedidos frequentes de baixo valor relativo e pedidos menos frequentes de alto valor. As duas primeiras dimensões preservam mais de 80%; a terceira preserva o restante.

## 7. Perfis e distribuição de valor

O radar usa médias de RFM log-padronizado reescaladas entre segmentos para [0,1]. Recência maior significa mais tempo sem comprar, portanto não representa melhor desempenho. O heatmap usa essas médias padronizadas nas cores e médias nas unidades originais nas anotações.

Os nomes são regras descritivas comparando médias dos clusters às medianas globais. Não são escores RFM clássicos de 1 a 5. Um nome de segmento não descreve todos os seus membros: médias podem ocultar heterogeneidade e serem influenciadas por atacadistas.

In [ ]:
display(perfil.round(2))
for nome in ['radar','heatmap','clientes','valor','receita']:
    grafico(nome)
display(Markdown(f"O maior segmento contém **{perfil.PercentualClientes.max():.1f}%** dos clientes. Uma partição muito desigual pode ter boas métricas e ainda oferecer pouca granularidade para campanhas; os resultados com k=3…8 permanecem disponíveis para avaliar esse trade-off."))

## 8. Relatório executivo, recomendações e limitações

O relatório seguinte é gerado com os resultados efetivamente calculados. Campanhas e prazos são propostas para teste; não há promessa de aumento de vendas nem estimativa de lucro sem custos e grupo de controle.

In [ ]:
relatorio = (out / 'relatorio_executivo.md').read_text(encoding='utf-8')
assert len(relatorio.split()) <= 1500
display(Markdown(relatorio))
print(f'Relatório: {len(relatorio.split())} palavras.')

## 9. Reprodutibilidade e entregáveis

`resultados/dashboard.html` contém os gráficos interativos com Plotly incorporado e funciona offline. As tabelas CSV agregadas permitem auditar perfis e escolhas. Os CSVs com clientes e probabilidades individuais são gerados localmente, sem versionamento no repositório público. `resumo.json` registra SHA-256 dos dados, semente e versão Python; `requirements-lock.txt` registra as bibliotecas.

O código e os entregáveis estão versionados no [GitHub](https://github.com/Cryovers/unsupervised-machine-earning-activity1). Consulte o README para execução e correspondência com o PDF.

Referências metodológicas: [scikit-learn — clustering](https://scikit-learn.org/stable/modules/clustering.html) e [Gaussian mixtures](https://scikit-learn.org/stable/modules/mixture.html).